# End-to-End RAG Evaluation: Llama 3.2 3B + All Retrievers


Evaluates all four retrievers in a full RAG pipeline:
- **Generator**: Llama 3.2 3B-Instruct (4-bit quantized)
- **Task**: PubMedQA yes/no/maybe classification
- **Metrics**: Accuracy + RAGAS faithfulness

In [2]:
!pip install torchao --upgrade -q
!pip install transformers peft datasets faiss-cpu ragas bitsandbytes accelerate rank_bm25 -q
print("Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 

In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import AutoModel
import faiss
from tqdm import tqdm
import json
import os
import gc
import re
from rank_bm25 import BM25Okapi

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"

CUDA available: False


## 0. HuggingFace Login

Required to access Llama 3.2 (gated model). Get your token from https://huggingface.co/settings/tokens

In [4]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HUGGING_FACE_HUB_TOKEN")

if hf_token is None:
    raise ValueError("Hugging Face token not found. Add it in Colab Secrets first.")

os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

login(token=hf_token)

print("Logged in.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in.


## 1. Mount Google Drive and Load All Results

In [5]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/cs505_project"


baseline = json.load(open(f"{DRIVE_DIR}/baseline_results_expanded.json"))
bm25_results = baseline["bm25"]
contriever_results = baseline["contriever"]
dapt_results = json.load(open(f"{DRIVE_DIR}/dapt_results.json"))
lora_results = json.load(open(f"{DRIVE_DIR}/lora_results.json"))

print("Retrieval results loaded:")
print(f"  BM25               Recall@5={bm25_results['Recall@5']:.3f}  MRR={bm25_results['MRR']:.3f}")
print(f"  Contriever         Recall@5={contriever_results['Recall@5']:.3f}  MRR={contriever_results['MRR']:.3f}")
print(f"  DAPT               Recall@5={dapt_results['Recall@5']:.3f}  MRR={dapt_results['MRR']:.3f}")
print(f"  LoRA               Recall@5={lora_results['Recall@5']:.3f}  MRR={lora_results['MRR']:.3f}")

Mounted at /content/drive
Retrieval results loaded:
  BM25               Recall@5=0.924  MRR=0.876
  Contriever         Recall@5=0.793  MRR=0.697
  DAPT               Recall@5=0.949  MRR=0.893
  LoRA               Recall@5=0.947  MRR=0.880


## 2. Load PubMedQA + Build Corpus

In [9]:
print("Loading PubMedQA...")
labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
distractors_raw = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

NUM_DISTRACTORS = 10000

queries = [ex["question"] for ex in labeled]
gold_abstracts = [" ".join(ex["context"]["contexts"]) for ex in labeled]
gold_labels = [ex["final_decision"] for ex in labeled]
gold_passage_ids = list(range(len(gold_abstracts)))

corpus = list(gold_abstracts)
for i, ex in enumerate(distractors_raw):
    if i >= NUM_DISTRACTORS:
        break
    corpus.append(" ".join(ex["context"]["contexts"]))

print(f"Queries: {len(queries)}, Corpus: {len(corpus)}")
labels, counts = np.unique(gold_labels, return_counts=True)
print(f"Label distribution: { {l: int(c) for l, c in zip(labels, counts)} }")

Loading PubMedQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

Queries: 1000, Corpus: 11000
Label distribution: {np.str_('maybe'): 110, np.str_('no'): 338, np.str_('yes'): 552}


## 3. Get Retrieved Passages for All Retrievers

In [11]:
def get_bm25_retrieved(queries, corpus, k=5):
    print("Building BM25 index...")
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    retrieved = []
    for query in tqdm(queries, desc="BM25"):
        scores = bm25.get_scores(query.lower().split())
        top_ids = np.argsort(scores)[::-1][:k].tolist()
        retrieved.append(top_ids)
    return retrieved

def get_dense_retrieved(corpus_emb_path, query_emb_path, k=5):
    print(f"Loading: {corpus_emb_path.split('/')[-1]}")
    corpus_emb = np.load(corpus_emb_path).astype(np.float32)
    query_emb = np.load(query_emb_path).astype(np.float32)
    faiss.normalize_L2(corpus_emb)
    faiss.normalize_L2(query_emb)
    index = faiss.IndexFlatIP(corpus_emb.shape[1])
    index.add(corpus_emb)
    _, retrieved_idx = index.search(query_emb, k)
    return [r.tolist() for r in retrieved_idx]

K = 5

print("Getting retrieved passages for all retrievers...")
bm25_retrieved = get_bm25_retrieved(queries, corpus, k=K)

contriever_retrieved = get_dense_retrieved(
    f"{DRIVE_DIR}/corpus_embeddings_expanded.npy",
    f"{DRIVE_DIR}/query_embeddings.npy", k=K)

dapt_retrieved = get_dense_retrieved(
    f"{DRIVE_DIR}/corpus_emb_dapt.npy",
    f"{DRIVE_DIR}/query_emb_dapt.npy", k=K)

lora_retrieved = get_dense_retrieved(
    f"{DRIVE_DIR}/corpus_emb_lora.npy",
    f"{DRIVE_DIR}/query_emb_lora.npy", k=K)

print("All retrieval done.")

Getting retrieved passages for all retrievers...
Building BM25 index...


BM25: 100%|██████████| 1000/1000 [01:36<00:00, 10.31it/s]


Loading: corpus_embeddings_expanded.npy
Loading: corpus_emb_dapt.npy
Loading: corpus_emb_lora.npy
All retrieval done.


## 4. Load Llama 3.2 3B-Instruct (4-bit quantized)

In [11]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_NAME} in 4-bit...")
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
llm_model.eval()

if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print("Model loaded.")

Loading meta-llama/Llama-3.2-3B-Instruct in 4-bit...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded.


## 5. RAG Inference Functions

In [14]:
def format_rag_prompt(query, passages):
    context = "\n\n".join([f"[Passage {i+1}]\n{p}" for i, p in enumerate(passages)])
    system_msg = (
        "You are a biomedical question answering assistant. "
        "Answer the question using ONLY the provided passages. "
        "Your answer must be exactly one word: yes, no, or maybe."
    )
    user_msg = f"""Passages:
{context}

Question: {query}

Answer (yes, no, or maybe):"""

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
    return llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

def extract_answer(text):
    text = text.lower().strip()
    for word in ["yes", "no", "maybe"]:
        if word in text:
            return word
    return "maybe"

def run_rag_evaluation(queries, corpus, retrieved_ids, gold_labels, desc="Evaluating"):
    predictions = []
    for i, (query, ret_ids, gold) in enumerate(
        tqdm(zip(queries, retrieved_ids, gold_labels), total=len(queries), desc=desc)):

        passages = [corpus[idx] for idx in ret_ids]
        prompt = format_rag_prompt(query, passages)

        inputs = llm_tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=2048
        ).to(device)

        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False,
                temperature=1.0,
                pad_token_id=llm_tokenizer.pad_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        answer_text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True)
        pred = extract_answer(answer_text)

        predictions.append({
            "query": query,
            "gold": gold,
            "pred": pred,
            "correct": pred == gold,
            "retrieved_ids": ret_ids,
            "raw_output": answer_text[:100]
        })
        del inputs, outputs

    accuracy = sum(p["correct"] for p in predictions) / len(predictions)
    return accuracy, predictions

print("RAG functions defined.")

RAG functions defined.


## 6. Run Evaluation - BM25

In [15]:
print("=== BM25 + Llama 3.2 3B ===")
bm25_acc, bm25_preds = run_rag_evaluation(
    queries[:200], corpus, bm25_retrieved[:200], gold_labels[:200], desc="BM25+RAG")
print(f"BM25 Accuracy: {bm25_acc:.4f}")
json.dump(bm25_preds, open("bm25_rag_preds.json", "w"), indent=2)

=== BM25 + Llama 3.2 3B ===


BM25+RAG: 100%|██████████| 200/200 [16:13<00:00,  4.87s/it]

BM25 Accuracy: 0.5850


## 6b. Run Evaluation - General Contriever

In [16]:
print("=== Contriever (general) + Llama 3.2 3B ===")
contriever_acc, contriever_preds = run_rag_evaluation(
    queries[:200], corpus, contriever_retrieved[:200], gold_labels[:200], desc="Contriever+RAG")
print(f"Contriever Accuracy: {contriever_acc:.4f}")
json.dump(contriever_preds, open("contriever_rag_preds.json", "w"), indent=2)

=== Contriever (general) + Llama 3.2 3B ===


Contriever+RAG: 100%|██████████| 200/200 [07:44<00:00,  2.32s/it]

Contriever Accuracy: 0.5750


## 6c. Run Evaluation - DAPT Contriever

In [17]:
print("=== DAPT Contriever + Llama 3.2 3B ===")
dapt_acc, dapt_preds = run_rag_evaluation(
    queries[:200], corpus, dapt_retrieved[:200], gold_labels[:200], desc="DAPT+RAG")
print(f"DAPT Accuracy: {dapt_acc:.4f}")
json.dump(dapt_preds, open("dapt_rag_preds.json", "w"), indent=2)

=== DAPT Contriever + Llama 3.2 3B ===


DAPT+RAG: 100%|██████████| 200/200 [15:22<00:00,  4.61s/it]

DAPT Accuracy: 0.5950


## 6d. Run Evaluation - LoRA Contriever

In [18]:
print("=== LoRA Contriever + Llama 3.2 3B ===")
lora_acc, lora_preds = run_rag_evaluation(
    queries[:200], corpus, lora_retrieved[:200], gold_labels[:200], desc="LoRA+RAG")
print(f"LoRA Accuracy: {lora_acc:.4f}")
json.dump(lora_preds, open("lora_rag_preds.json", "w"), indent=2)

=== LoRA Contriever + Llama 3.2 3B ===


LoRA+RAG: 100%|██████████| 200/200 [14:44<00:00,  4.42s/it]

LoRA Accuracy: 0.6450


## 7. RAGAS Faithfulness (50-query sample per retriever)

In [19]:
from transformers import pipeline
from tqdm import tqdm
import json

# Reload predictions
bm25_preds = json.load(open("bm25_rag_preds.json"))
contriever_preds = json.load(open("contriever_rag_preds.json"))
dapt_preds = json.load(open("dapt_rag_preds.json"))
lora_preds = json.load(open("lora_rag_preds.json"))
print("Predictions loaded.")

print("Loading NLI model...")
nli_pipe = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=0 if torch.cuda.is_available() else -1
)

# Check what labels the model actually uses
test = nli_pipe("The sky is blue. [SEP] yes", truncation=True, max_length=512)
print(f"NLI model label example: {test}")
# Use whichever label corresponds to entailment (usually highest score label)

Predictions loaded.
Loading NLI model...


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model label example: [{'label': 'neutral', 'score': 0.9754255414009094}]


In [22]:
# Debug: inspect raw NLI scores for a few examples
for i in range(3):
    pred = bm25_preds[i]
    passages = [corpus[idx] for idx in bm25_retrieved[:200][i]]
    context = " ".join(passages)[:1000]
    hypothesis = f"The answer to '{pred['query']}' is {pred['pred']}."

    results = nli_pipe(
        f"{context} [SEP] {hypothesis}",
        truncation=True,
        max_length=512,
        top_k=3
    )
    print(f"Query: {pred['query'][:80]}...")
    print(f"Answer: {pred['pred']}, Gold: {pred['gold']}")
    print(f"Scores: {results}")
    print()

Query: Do mitochondria play a role in remodelling lace plant leaves during programmed c...
Answer: yes, Gold: yes
Scores: [{'label': 'neutral', 'score': 0.9982445240020752}, {'label': 'contradiction', 'score': 0.0011968136532232165}, {'label': 'entailment', 'score': 0.0005586178158409894}]

Query: Landolt C and snellen e acuity: differences in strabismus amblyopia?...
Answer: yes, Gold: no
Scores: [{'label': 'neutral', 'score': 0.8674925565719604}, {'label': 'entailment', 'score': 0.09176494181156158}, {'label': 'contradiction', 'score': 0.04074246808886528}]

Query: Syncope during bathing in infants, a pediatric form of water-induced urticaria?...
Answer: no, Gold: yes
Scores: [{'label': 'neutral', 'score': 0.9981921315193176}, {'label': 'contradiction', 'score': 0.001573932939209044}, {'label': 'entailment', 'score': 0.00023398092889692634}]



## 8. Full Results Summary

In [23]:
# Hard-code accuracy from completed runs
bm25_acc       = 0.5850
contriever_acc = 0.5750
dapt_acc       = 0.5950
lora_acc       = 0.6450

bm25_results       = {"Recall@5": 0.924, "Recall@20": 0.953, "MRR": 0.876}
contriever_results = {"Recall@5": 0.793, "Recall@20": 0.942, "MRR": 0.697}
dapt_results       = {"Recall@5": 0.949, "Recall@20": 0.982, "MRR": 0.893}
lora_results       = {"Recall@5": 0.947, "Recall@20": 0.981, "MRR": 0.880}

print("=" * 65)
print("FULL RESULTS (200-query sample, Llama 3.2 3B generator)")
print("=" * 65)
print(f"\n{'Retriever':<25} {'Recall@5':<12} {'MRR':<12} {'Accuracy'}")
print("-" * 60)
print(f"{'BM25':<25} {bm25_results['Recall@5']:<12.3f} {bm25_results['MRR']:<12.3f} {bm25_acc:.3f}")
print(f"{'Contriever (general)':<25} {contriever_results['Recall@5']:<12.3f} {contriever_results['MRR']:<12.3f} {contriever_acc:.3f}")
print(f"{'Contriever + DAPT':<25} {dapt_results['Recall@5']:<12.3f} {dapt_results['MRR']:<12.3f} {dapt_acc:.3f}")
print(f"{'Contriever + LoRA':<25} {lora_results['Recall@5']:<12.3f} {lora_results['MRR']:<12.3f} {lora_acc:.3f}")

print("\n\nLaTeX rows (Table 2):")
print(f"BM25 & {bm25_results['Recall@5']:.3f} & {bm25_results['MRR']:.3f} & {bm25_acc:.3f} \\\\")
print(f"Contriever & {contriever_results['Recall@5']:.3f} & {contriever_results['MRR']:.3f} & {contriever_acc:.3f} \\\\")
print(f"Contriever + DAPT & {dapt_results['Recall@5']:.3f} & {dapt_results['MRR']:.3f} & {dapt_acc:.3f} \\\\")
print(f"Contriever + LoRA & {lora_results['Recall@5']:.3f} & {lora_results['MRR']:.3f} & {lora_acc:.3f} \\\\")

import json
all_results = {
    "bm25":       {"retrieval": bm25_results, "accuracy": bm25_acc},
    "contriever": {"retrieval": contriever_results, "accuracy": contriever_acc},
    "dapt":       {"retrieval": dapt_results, "accuracy": dapt_acc},
    "lora":       {"retrieval": lora_results, "accuracy": lora_acc},
}
json.dump(all_results, open("all_results.json", "w"), indent=2)
print("\nSaved to all_results.json")

FULL RESULTS (200-query sample, Llama 3.2 3B generator)

Retriever                 Recall@5     MRR          Accuracy
------------------------------------------------------------
BM25                      0.924        0.876        0.585
Contriever (general)      0.793        0.697        0.575
Contriever + DAPT         0.949        0.893        0.595
Contriever + LoRA         0.947        0.880        0.645


LaTeX rows (Table 2):
BM25 & 0.924 & 0.876 & 0.585 \\
Contriever & 0.793 & 0.697 & 0.575 \\
Contriever + DAPT & 0.949 & 0.893 & 0.595 \\
Contriever + LoRA & 0.947 & 0.880 & 0.645 \\

Saved to all_results.json


## 9. Error Analysis Table

Categorize retrieval failures by type for BM25 vs DAPT feedback.

In [24]:
def categorize_failure(query):
    q = query.lower().strip()
    if len(q.split()) <= 6:
        return "Short/vague query"
    elif any(w in q for w in ["carcinoma", "hepatocellular", "adenocarcinoma",
                               "thyroidectomy", "immunohistochemistry",
                               "pharmacokinetic", "polymorphism", "genotype", "cytokine"]):
        return "Highly specialized terminology"
    elif q.startswith(("is ", "are ", "does ", "do ", "was ", "were ", "can ", "could ")):
        return "General yes/no question"
    else:
        return "Complex multi-concept query"

# Reload full top-20 lists for BM25 and DAPT
print("Building BM25 top-20 for error analysis...")
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25_obj = BM25Okapi(tokenized_corpus)
bm25_full = []
for q in tqdm(queries, desc="BM25 full"):
    scores = bm25_obj.get_scores(q.lower().split())
    bm25_full.append(np.argsort(scores)[::-1][:20].tolist())

print("Loading DAPT embeddings for error analysis...")
dapt_emb_c = np.load(f"{DRIVE_DIR}/corpus_emb_dapt.npy").astype(np.float32)
dapt_emb_q = np.load(f"{DRIVE_DIR}/query_emb_dapt.npy").astype(np.float32)
faiss.normalize_L2(dapt_emb_c); faiss.normalize_L2(dapt_emb_q)
idx = faiss.IndexFlatIP(dapt_emb_c.shape[1])
idx.add(dapt_emb_c)
_, dapt_full_idx = idx.search(dapt_emb_q, 20)
dapt_full = [r.tolist() for r in dapt_full_idx]

failure_types = [
    "Short/vague query",
    "Highly specialized terminology",
    "General yes/no question",
    "Complex multi-concept query"
]

bm25_fail = {t: 0 for t in failure_types}
dapt_fail = {t: 0 for t in failure_types}

for i, (query, gold_id) in enumerate(zip(queries, gold_passage_ids)):
    ftype = categorize_failure(query)
    if gold_id not in bm25_full[i][:5]:
        bm25_fail[ftype] += 1
    if gold_id not in dapt_full[i][:5]:
        dapt_fail[ftype] += 1

print(f"\n{'Failure Type':<35} {'BM25':>8} {'DAPT':>8}")
print("-" * 53)
for t in failure_types:
    print(f"{t:<35} {bm25_fail[t]:>8} {dapt_fail[t]:>8}")
print(f"{'Total':<35} {sum(bm25_fail.values()):>8} {sum(dapt_fail.values()):>8}")

json.dump({"bm25": bm25_fail, "dapt": dapt_fail}, open("error_analysis.json", "w"), indent=2)
print("\nError analysis saved.")

Building BM25 top-20 for error analysis...


BM25 full: 100%|██████████| 1000/1000 [01:40<00:00,  9.97it/s]


Loading DAPT embeddings for error analysis...

Failure Type                            BM25     DAPT
-----------------------------------------------------
Short/vague query                          7        4
Highly specialized terminology             4        3
General yes/no question                   36       30
Complex multi-concept query               29       14
Total                                     76       51

Error analysis saved.


## 10. Save Everything to Google Drive

In [25]:
import shutil

files_to_save = [
    "all_results.json",
    "error_analysis.json",
    "bm25_rag_preds.json",
    "contriever_rag_preds.json",
    "dapt_rag_preds.json",
    "lora_rag_preds.json",
]

for f in files_to_save:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print(f"Saved: {f}")

print(f"\nAll files saved to {DRIVE_DIR}")

Saved: all_results.json
Saved: error_analysis.json
Saved: bm25_rag_preds.json
Saved: contriever_rag_preds.json
Saved: dapt_rag_preds.json
Saved: lora_rag_preds.json

All files saved to /content/drive/MyDrive/cs505_project
